# Denoising autoencoder con masking mediante Dropout, DS2OS

Anula una fracción de las columnas de entrada con una capa Dropout, activa solo en entrenamiento. Dropout reescala además las columnas que no anula. El nivel 0 es el control sin corrupción.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

In [2]:
DATA_PATH = Path.home() / ".cache/kagglehub/datasets/libamariyam/ds2os-dataset/versions/1/DS2OS.csv"

df = pd.read_csv(DATA_PATH).drop(columns=['timestamp'])
df['accessedNodeType'] = df['accessedNodeType'].fillna('missing')
df['value'] = df['value'].replace({'true': 1, 'false': 0, 'twenty': 20, 'none': 0})
df['value'] = pd.to_numeric(df['value'], errors='coerce').fillna(0)
print(df.shape)

(357952, 12)


In [3]:
from sklearn.model_selection import train_test_split

CAT_COLS = ['sourceID', 'sourceAddress', 'sourceType', 'sourceLocation',
            'destinationServiceAddress', 'destinationServiceType', 'destinationLocation',
            'accessedNodeAddress', 'accessedNodeType', 'operation']

y = (df['normality'] != 'normal').astype(int).values
X = pd.get_dummies(df.drop(columns=['normality']), columns=CAT_COLS)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

scaler = MinMaxScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)
X_train_normal = X_train[y_train == 0]

print(f"X_train: {X_train.shape}, X_train_normal: {X_train_normal.shape}, X_test: {X_test.shape}")

X_train: (286361, 505), X_train_normal: (278347, 505), X_test: (71591, 505)


In [4]:
def build_dae_masking(input_dim, hidden_units, mask_rate):
    inputs = keras.Input(shape=(input_dim,))
    x = layers.Dropout(mask_rate)(inputs)
    for units in hidden_units[:-1]:
        x = layers.Dense(units, activation='relu')(x)
    x = layers.Dense(hidden_units[-1], activation='relu')(x)
    for units in reversed(hidden_units[:-1]):
        x = layers.Dense(units, activation='relu')(x)
    outputs = layers.Dense(input_dim, activation='sigmoid')(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='mse')
    return model


def reconstruction_error(model, X):
    return np.mean(np.square(X - model.predict(X, verbose=0)), axis=1)


HIDDEN_UNITS = [128, 64, 16]
MASK_RATES = [0.0, 0.1, 0.2, 0.3]  # 0.0 = control sin corrupción

results = {}
for rate in MASK_RATES:
    tf.keras.utils.set_random_seed(SEED)
    model = build_dae_masking(X_train.shape[1], HIDDEN_UNITS, rate)
    history = model.fit(
        X_train_normal, X_train_normal,
        epochs=100,
        batch_size=256,
        validation_split=0.1,
        callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)],
        verbose=0,
    )
    auc = roc_auc_score(y_test, reconstruction_error(model, X_test))
    results[rate] = {'model': model, 'auc': auc, 'epochs': len(history.history['loss'])}
    print(f"mask_rate={rate}: epocas={results[rate]['epochs']}, ROC-AUC={auc:.4f}")

mask_rate=0.0: epocas=99, ROC-AUC=0.9211


mask_rate=0.1: epocas=100, ROC-AUC=0.9269


mask_rate=0.2: epocas=100, ROC-AUC=0.8848


mask_rate=0.3: epocas=100, ROC-AUC=0.8968


In [5]:
best_rate = max(results, key=lambda k: results[k]['auc'])
best_model = results[best_rate]['model']
print(f"Mejor variante: mask_rate={best_rate} (ROC-AUC={results[best_rate]['auc']:.4f})")
print(f"Control (mask_rate=0.0, mismo codigo, sin corrupcion): ROC-AUC={results[0.0]['auc']:.4f}")
print(f"Diferencia mejor DAE-masking vs control: {results[best_rate]['auc'] - results[0.0]['auc']:+.4f}")

threshold = np.percentile(reconstruction_error(best_model, X_train_normal), 95)
y_pred = (reconstruction_error(best_model, X_test) > threshold).astype(int)

print()
print(classification_report(y_test, y_pred, target_names=['normal', 'ataque']))
print(confusion_matrix(y_test, y_pred))
print(f"ROC-AUC (mejor variante DAE-masking): {results[best_rate]['auc']:.4f}")

Mejor variante: mask_rate=0.1 (ROC-AUC=0.9269)
Control (mask_rate=0.0, mismo codigo, sin corrupcion): ROC-AUC=0.9211
Diferencia mejor DAE-masking vs control: +0.0059



              precision    recall  f1-score   support

      normal       0.99      0.99      0.99     69588
      ataque       0.76      0.73      0.74      2003

    accuracy                           0.99     71591
   macro avg       0.88      0.86      0.87     71591
weighted avg       0.99      0.99      0.99     71591

[[69129   459]
 [  548  1455]]
ROC-AUC (mejor variante DAE-masking): 0.9269
